<a href="https://colab.research.google.com/github/rahul02500/Practicepython/blob/main/Building%20your%20First%20AI%20Agent%20with%20Autogen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install autogen-agentchat==0.2.38 pandas matplotlib seaborn


In [2]:
import autogen
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
from autogen.coding import LocalCommandLineCodeExecutor
from autogen import config_list_from_json

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Writing OAI_CONFIG_LIST.json


In [3]:
from google.colab import userdata

# Replace 'OPENAI_API_KEY' with the exact name you used in Colab Secrets
Openai_key = userdata.get('OpenAI')

In [4]:
import os

os.environ['OPENAI_API_KEY'] = Openai_key

In [5]:
import json
import os
from autogen import config_list_from_json

# Define the config list structure using the API key from environment variables
config_list_data = [
    {
        "model": "gpt-4",
        "api_key": os.environ["OPENAI_API_KEY"]
    },
    {
        "model": "gpt-3.5-turbo",
        "api_key": os.environ["OPENAI_API_KEY"]
    }
]

# Write the valid JSON content to OAI_CONFIG_LIST.json
with open("OAI_CONFIG_LIST.json", "w") as f:
    json.dump(config_list_data, f, indent=4)

# Now, load the correctly formatted config list
config_list = config_list_from_json("OAI_CONFIG_LIST.json")

In [6]:
llm_config = {
    "cache_seed": 42, # Added cache_seed as it was in the original llm_config
    "temperature": 0,
    "config_list": config_list, # Use the config_list that was just loaded
    "timeout": 120,
}

In [7]:
data_prep_agent = AssistantAgent(
    name="DataPrepAgent",
    llm_config=llm_config,
    system_message="""
    You are a Data Preparation Agent.
    Responsibilities:
    - Handle missing values
    - Remove duplicates
    - Fix data types
    - Prepare dataset for analysis
    Only provide clean Python code.
    """
)

In [8]:
eda_agent = AssistantAgent(
    name="EDAAgent",
    llm_config=llm_config,
    system_message="""
    You are an EDA Agent.
    Responsibilities:
    - Generate summary statistics
    - Identify patterns, correlations
    - Create visualizations
    Use pandas, matplotlib, seaborn.
    Provide Python code and insights.
    """
)

In [9]:
report_agent = AssistantAgent(
    name="ReportAgent",
    llm_config=llm_config,
    system_message="""
    You are a Report Generator Agent.
    Responsibilities:
    - Create structured EDA report
    - Include summary, insights, findings
    - Make output business-friendly
    No code, only formatted report.
    """
)

In [10]:
critic_agent = AssistantAgent(
    name="CriticAgent",
    llm_config=llm_config,
    system_message="""
    You are a Critic Agent.
    Responsibilities:
    - Review outputs of other agents
    - Suggest improvements
    - Ensure clarity and correctness
    """
)

In [11]:
executor = UserProxyAgent(
    name="Executor",
    system_message="Executes Python code and returns results.",
    code_execution_config={
        "executor": LocalCommandLineCodeExecutor()
    }
)

In [12]:
admin = UserProxyAgent(
    name="Admin",
    system_message="""
    You are Admin.
    Approve plan and ensure workflow runs properly.
    """,
    code_execution_config=False
)

In [13]:
groupchat = GroupChat(
    agents=[
        admin,
        data_prep_agent,
        eda_agent,
        report_agent,
        critic_agent,
        executor
    ],
    messages=[],
    max_round=10
)

manager = GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config
)

In [ ]:
task = """
Perform complete EDA on dataset 'data.csv'.

Steps:
1. Clean the data
2. Perform EDA analysis
3. Generate insights and visualizations
4. Create final structured report
5. Critic review

Ensure output is clear and business-ready.
"""

admin.initiate_chat(manager, message=task)

Admin (to chat_manager):


Perform complete EDA on dataset 'data.csv'.

Steps:
1. Clean the data
2. Perform EDA analysis
3. Generate insights and visualizations
4. Create final structured report
5. Critic review

Ensure output is clear and business-ready.


--------------------------------------------------------------------------------

Next speaker: DataPrepAgent

DataPrepAgent (to chat_manager):

As a Data Preparation Agent, I can help you with the first step of your request, which is cleaning the data. Here's how you can do it using Python and pandas:

```python
import pandas as pd

# Load the dataset
df = pd.read_csv('data.csv')

# Handle missing values
# If the data is numerical you can fill missing data with mean or median
df = df.fillna(df.mean())

# If the data is categorical you can fill missing data with mode
df = df.fillna(df.mode().iloc[0])

# Remove duplicates
df = df.drop_duplicates()

# Fix data types
# This is highly dependent on the columns in your dataset
# Here is a